# AI Music Creator — on a rented GPU

This repo is the **tool**. A clone of it is complete on its own: the presets in
`presets/` work, `night/collections/example.py` is a batch you can render, and
you can write your own beside both.

If you also have a **private workspace** — your own presets, lyrics and
collections in a second repository — step 4 clones it in. Skipping step 4 is
fine and always was: the tool then uses this repo as its own workspace.

Two things are missing from any fresh runtime, being the two `.gitignore` keeps
out of git entirely:

| | | |
|---|---|---|
| `engine/` | the upstream [ACE-Step 1.5](https://github.com/ace-step/ACE-Step-1.5) clone | ~5 GB of wheels |
| `engine/checkpoints/` | the weights | ~8 GB |

Step 5 fetches both, in about ten minutes, and again on every new machine Colab
gives you — which is why step 2 mounts Drive, so the songs survive that.

> **Runtime → Change runtime type → GPU** first. A free T4 renders a
> three-minute take in well under a minute, against roughly nine on a MacBook.

**Only step 3 is mandatory.** Every cell after it recomputes what it needs from
what is on disk, so cells can be skipped, re-run, or run out of order.

## 1 · What are we running on

In [ ]:
#@title Check the GPU { display-mode: "form" }
# Informational. Step 3 works this out again for itself, so skipping this
# changes nothing except that you do not get to read it.
import torch

if not torch.cuda.is_available():
    print("No GPU. Runtime -> Change runtime type -> GPU, then re-run.")
else:
    major, minor = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"{torch.cuda.get_device_name(0)}  —  {vram:.0f} GiB, "
          f"compute capability {major}.{minor}")
    # nano-vllm (the `vllm` backend, aimc's default on CUDA) wants bfloat16,
    # which arrived with Ampere — capability 8.0. On an older card the engine
    # would try it, fail, and fall back to PyTorch anyway.
    print("LM backend: " + ("vllm" if (major, minor) >= (8, 0)
                            else "pt — this card predates bfloat16"))

## 2 · Somewhere for the songs to land

Colab throws the machine away, and a long queue outlasts a free session. Step 7
copies each finished take to Drive as it lands, so a disconnect costs the take
in flight and nothing else.

Skip this cell to render into the runtime and download a zip at the end.

In [ ]:
#@title Mount Google Drive { display-mode: "form" }
DRIVE_FOLDER = "AI_Music_Creator"  #@param {type:"string"}

from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
Path("/content/drive/MyDrive", DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
print(f"takes and ledger will be copied to: MyDrive/{DRIVE_FOLDER}")

## 3 · The tool  *(required)*

A few megabytes. This cell also defines `context()`, which every later cell
calls to find out where things are — so the rest of the notebook does not depend
on which cells you ran, or in what order.

In [ ]:
#@title Clone the tool, and set up the notebook { display-mode: "form" }
REPO_URL = "https://github.com/Netajam/AI_Music_Creator.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import contextlib
import json
import os
import re
import shlex
import subprocess
import sys
import threading
import time
from pathlib import Path
from types import SimpleNamespace
from typing import Any


def git(into: Path, *args: str) -> subprocess.CompletedProcess:
    """git, with its output captured so it can reach the cell.

    Everything a subprocess writes to the kernel's file descriptors is lost in
    Jupyter — which is how a pull that failed and a pull that did nothing came
    to look identical from here.
    """
    return subprocess.run(["git", "-C", str(into), *args],
                          capture_output=True, text=True)


def clone_or_pull(url: str, into: Path, branch: str) -> None:
    """Fetch, or fast-forward what is already here, and say which.

    --ff-only, and it is not timidity: a merge conflict resolved inside a
    machine that is about to be deleted is work thrown away. Edits happen
    elsewhere; this end only ever catches up to them.
    """
    if not (into / ".git").exists():
        done = subprocess.run(["git", "clone", "--branch", branch, url, str(into)],
                              capture_output=True, text=True)
        if done.returncode:
            raise SystemExit(f"clone failed:\n{done.stderr.strip()}")
        print(f"cloned {into.name} at {git(into, 'rev-parse', '--short', 'HEAD').stdout.strip()}")
        return

    before = git(into, "rev-parse", "HEAD").stdout.strip()
    fetched = git(into, "fetch", "origin", branch)
    if fetched.returncode:
        raise SystemExit(f"fetch failed:\n{fetched.stderr.strip()}")

    for step in (("checkout", branch), ("merge", "--ff-only", f"origin/{branch}")):
        done = git(into, *step)
        if done.returncode:
            why = (done.stderr or done.stdout).strip().splitlines()[:4]
            print(f"could not {step[0]} — {into.name} is left as it was:")
            print("\n".join("  " + w for w in why))
            print(f"  This end only ever catches up. To discard whatever is in the "
                  f"way:\n    !git -C {into} reset --hard origin/{branch}")
            return

    after = git(into, "rev-parse", "HEAD").stdout.strip()
    if before == after:
        print(f"{into.name}: already up to date at {after[:7]}")
    else:
        n = git(into, "rev-list", "--count", f"{before}..{after}").stdout.strip()
        print(f"{into.name}: updated {before[:7]} -> {after[:7]}  ({n} commit(s))")


def committed(into: Path, path: str) -> str:
    """The hash git has for one file at HEAD, or "" if it cannot say."""
    got = subprocess.run(["git", "-C", str(into), "rev-parse", f"HEAD:{path}"],
                         capture_output=True, text=True)
    return got.stdout.strip() if got.returncode == 0 else ""


NOTEBOOK = "colab/AI_Music_Creator.ipynb"
ROOT = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

was = committed(ROOT, NOTEBOOK) if (ROOT / ".git").exists() else ""
clone_or_pull(REPO_URL, ROOT, BRANCH)
now = committed(ROOT, NOTEBOOK)
os.chdir(ROOT)

# A pull updates the repo on disk. It does not update the cells you are running:
# Colab loaded those when you opened the notebook, and it is the one file in
# here that a pull cannot deliver to you. Say so, rather than let the old cells
# run against the new code.
if was and now and was != now:
    print("\n" + "─" * 68)
    print("  This pull changed the notebook itself.")
    print("  You are still running the cells Colab loaded when you opened it.")
    print("  Re-open it to get the new ones (your runtime and downloads survive):")
    print("  https://colab.research.google.com/github/Netajam/AI_Music_Creator"
          "/blob/main/colab/AI_Music_Creator.ipynb")
    print("─" * 68 + "\n")
# uv installs to ~/.local/bin, and every wrapper in this repo runs through it.
if str(Path.home() / ".local/bin") not in os.environ["PATH"]:
    os.environ["PATH"] = f"{Path.home()}/.local/bin:{os.environ['PATH']}"


def context() -> SimpleNamespace:
    """Everything the cells below need, worked out fresh each time.

    A notebook is not a script. Cells get skipped, re-run, and run out of
    order, and a global left behind by one of them is a cell that only works
    if you happened to run another. So nothing here is remembered: the
    workspace is re-read from disk (step 4 may have written `.workspace` since),
    the GPU is re-detected, and Drive counts as mounted only if it is.
    """
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    # Dropped from the module cache, because `.workspace` may have changed and
    # aimc.workspace answers that question once, at import.
    for stale in [m for m in sys.modules if m.startswith("aimc")]:
        del sys.modules[stale]
    from aimc.workspace import LYRICS, NIGHT, PRESETS, REFS, SONGS, WORKSPACE

    extra = "--device cuda"
    import torch
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0) < (8, 0):
        extra += " --backend pt"

    # Step 2's folder name if that cell was run, its default otherwise.
    folder = globals().get("DRIVE_FOLDER", "AI_Music_Creator")
    mounted = Path("/content/drive/MyDrive")
    drive_dir = mounted / folder if mounted.is_dir() else None

    return SimpleNamespace(ROOT=ROOT, WORKSPACE=WORKSPACE, NIGHT=NIGHT,
                           PRESETS=PRESETS, LYRICS=LYRICS, SONGS=SONGS,
                           REFS=REFS, EXTRA=extra, DRIVE=drive_dir)


def to_drive(c: SimpleNamespace) -> None:
    """Copy what has landed out of the machine that is going to be deleted.

    One direction only, workspace -> Drive. `.run-*` is the engine's staging
    folder for a take in flight and `.started-*` the runner's marker: copying
    either would copy a half-written wav and call it a song.
    """
    if c.DRIVE is None:
        return
    c.DRIVE.mkdir(parents=True, exist_ok=True)
    if c.SONGS.exists():
        (c.DRIVE / "songs").mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", "--exclude", ".run-*", "--exclude",
                        ".started-*", f"{c.SONGS}/", str(c.DRIVE / "songs")],
                       check=False)
    for f in (c.NIGHT / "ledger.tsv", c.NIGHT / "worker-console.log"):
        if f.exists():
            subprocess.run(["cp", str(f), str(c.DRIVE / f.name)], check=False)


def finished_takes(c: SimpleNamespace) -> list[Path]:
    """Every finished take under songs/, oldest last-modified first.

    The filter is the whole point. `.run-*` is the engine's staging folder for
    a take in flight, and rglob walks into it like any other directory: while a
    render is running, the half-written wav in there is the newest file under
    songs/. Worse, it outlives the run — step 8 sends SIGTERM, which kills the
    worker where it stands rather than through the cleanup at the end of it, so
    every stopped render leaves a staging folder behind whose wav then outranks
    every real take, permanently. A wav is a take only once it has been moved
    out of staging and given its final name.
    """
    if not c.SONGS.exists():
        return []
    return sorted(
        (p for p in c.SONGS.rglob("*.wav")
         if not any(part.startswith(".") for part in p.relative_to(c.SONGS).parts)),
        key=lambda p: p.stat().st_mtime)


# Suggested shapes for the prompt below. They are suggestions and not a rule:
# night/build.py accepts 3 to 12 sections, deliberately, because treating eight
# as the only legal shape is what made the first hundred songs sound like one
# songwriter. A mantra or a one-riff track is shorter and still right.
STRUCTURES = {
    "pop": ("Intro", "Verse", "Chorus", "Verse", "Chorus", "Bridge", "Chorus", "Outro"),
    "electro": ("Intro", "Verse", "Build", "Drop", "Breakdown", "Build", "Drop", "Outro"),
    "mantra": ("Intro", "Verse", "Chorus", "Verse", "Chorus", "Outro"),
}

# Syllables per second, inside the 0.8–5.0 band night/build.py enforces. The
# omad take that was liked sits at 2.7 and a fast deejay flow reaches about 4.
DENSITY = {"sung": (2.0, 3.2), "spoken": (3.2, 4.5)}


def night_build() -> Any:
    """night/build.py, loaded from its path — for its lyric checker.

    Reimplementing that check here would be two rulebooks with one job, and the
    copy would drift. It also already knows two things a fresh attempt gets
    wrong: it counts **syllables**, not characters, because a character band is
    a Latin-script measure that rejected perfectly ordinary lyrics in other
    scripts; and it reads `[Verse 2]` as a Verse rather than as an invented tag.

    Loaded by path on every call rather than imported once, because `context()`
    drops aimc from the module cache whenever the workspace may have changed,
    and this module imports aimc.workspace at its top.
    """
    import importlib.util
    spec = importlib.util.spec_from_file_location("night_build", ROOT / "night" / "build.py")
    assert spec and spec.loader
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def lyrics_prompt(theme: str, language: str, structure: str, duration: int,
                  delivery: str) -> str:
    """The rules of docs/recipes.md, written out for a chat model.

    Generation is the commodity half of this; the rules are the half this repo
    learned by rendering. A model left to itself invents `[Pre-Chorus]` and
    `[guitar solo]`, which the 5Hz LM reads as sections so the music stops, and
    rewords the chorus every pass for variety — which is the one thing that
    stops a song sticking. Neither shows up until the take has been paid for.
    """
    shape = STRUCTURES[structure]
    lo, hi = DENSITY[delivery]
    refrain = "Drop" if "Drop" in shape else "Chorus"
    # A syllable count is the honest target but not a usable instruction; lines
    # of a singable length are the same number said in a way a model can aim at.
    syllables = int(duration * (lo + hi) / 2)
    return f"""\
Write song lyrics in {language}.

Theme: {theme}

Follow every rule below exactly. They come from a specific music model, and
breaking one spoils the render.

1. Structure — these {len(shape)} sections, in this order, nothing else:
{chr(10).join(f"   [{tag}]" for tag in shape)}

2. Bracketed tags: ONLY the ones above, spelled exactly as shown. Never write
   [Pre-Chorus], [Hook], [Refrain], [Guitar solo], [Instrumental], or a stage
   direction like [beat drops] or [voice alone]. The model reads anything in
   brackets as a new section, so an invented cue makes the music stop.

3. Every [{refrain}] must contain THE SAME WORDS, copied character for
   character. Do not vary it. That repetition is what makes a song memorable.

4. The verses must differ from each other.

5. Length: about {syllables} syllables over the whole lyric — roughly
   {syllables // 9} lines of eight to ten syllables. The track is {duration}
   seconds and the delivery is {delivery}, which is {lo}–{hi} syllables per
   second. Too many and the voice recites without room to breathe.

6. Whole lines, one per bar. Not shouted fragments.

7. [Intro] and [Outro] may be left empty — the tag on its own line with nothing
   under it. That is normal, and often better than filling them.

Return only the lyrics, starting with [{shape[0]}]. No commentary, no code fence."""


def engine_ready(c: SimpleNamespace) -> bool:
    """Whether step 5 has run on this machine, said before a tool has to say it.

    `./song` refuses without the checkpoints and exits 2 in about a tenth of a
    second, which reads exactly like a broken command rather than a missing
    install — so check here, where the answer is "run step 5" rather than the
    Mac instructions the CLI prints.
    """
    missing = [name for name, path in (("dependencies", c.ROOT / "engine/.venv"),
                                       ("weights", c.ROOT / "engine/checkpoints"))
               if not path.is_dir()]
    if missing:
        print(f"Not installed on this machine yet: {', '.join(missing)}.")
        print("Run step 5 first — about ten minutes.")
    return not missing


def bar(done: float, total: float, width: int = 32) -> str:
    """A text bar. Text, because it reads the same in either Colab theme."""
    frac = 0.0 if not total else max(0.0, min(1.0, done / total))
    filled = int(width * frac)
    return f"[{'█' * filled}{'·' * (width - filled)}] {frac * 100:3.0f}%"


def clock(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"


class Line:
    """One output line that rewrites itself, or plain prints if there is no UI."""

    def __init__(self) -> None:
        try:
            from IPython.display import Pretty, display
            self._pretty = Pretty
            self._handle = display(Pretty(""), display_id=True)
        except Exception:
            self._handle = None

    def __call__(self, text: str) -> None:
        if self._handle is None:
            print(text, flush=True)
        else:
            self._handle.update(self._pretty(text))


def expected_seconds(c: SimpleNamespace, steps: int) -> float | None:
    """How long a take has taken on *this* machine, per the engine's own notes.

    `handler.py` writes a record per generation to
    `.cache/acestep/progress_estimates.json` — device, infer_steps, seconds per
    step. It builds that path from its own idea of the project root, which is
    upstream's business and re-cloned on every run, so both plausible places are
    checked rather than one of them assumed.

    Nothing here estimates anything the engine has not already measured. On a
    fresh runtime it returns None, and the display says elapsed only rather than
    inventing a percentage.
    """
    records: list[dict] = []
    for notes in (c.ROOT / ".cache/acestep/progress_estimates.json",
                  c.ROOT / "engine/.cache/acestep/progress_estimates.json"):
        if notes.is_file():
            # An unreadable or half-written cache is not worth an exception:
            # the honest answer is then "no estimate", which is already handled.
            with contextlib.suppress(OSError, ValueError):
                records += json.loads(notes.read_text()).get("records", [])
    if not records:
        return None
    device = "cuda" if "cuda" in c.EXTRA else "mps"
    per_step = sorted(r["per_step_sec"] for r in records
                      if r.get("device") == device and r.get("infer_steps") == steps)
    if not per_step:
        return None
    return per_step[len(per_step) // 2] * steps      # median, times the steps


def run_song(argv: list[str], expect: float | None = None) -> int:
    """Run a wrapper, show its output *in this cell*, and say how it is going.

    A subprocess inherits the kernel's file descriptors, and in Jupyter those do
    not go to the cell: only Python-level writes to sys.stdout are captured. So
    a plain subprocess.run() generates a song and shows you nothing at all,
    including when it fails. Read the pipe and print it line by line instead.

    The status line is elapsed time, and a bar only when `expect` came from a
    real measurement. The diffusion loop reports no steps — tqdm covers the
    language model alone, and turns itself off when stderr is not a terminal —
    so a percentage here would be a guess dressed as progress.
    """
    print(" ".join(shlex.quote(a) for a in argv) + "\n", flush=True)
    started = time.time()
    line, stop = Line(), threading.Event()

    def tick() -> None:
        while not stop.is_set():
            elapsed = time.time() - started
            if expect:
                line(f"  {bar(elapsed, expect)}  {clock(elapsed)} elapsed, "
                     f"~{clock(expect)} expected")
            else:
                line(f"  rendering — {clock(elapsed)} elapsed")
            stop.wait(1)

    ticker = threading.Thread(target=tick, daemon=True)
    ticker.start()
    proc = subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    try:
        for out in proc.stdout:
            print(out, end="")
        code = proc.wait()
    finally:
        stop.set()
        ticker.join(timeout=2)

    line(f"  {clock(time.time() - started)} total")
    if code:
        print(f"\n./song exited {code} — the reason is above.")
    return code


c = context()
print(subprocess.run(["git", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)
print(f"repo      : {c.ROOT}")
print(f"workspace : {c.WORKSPACE}"
      f"{'  (this repo — no private workspace)' if c.WORKSPACE == c.ROOT else ''}")
print(f"drive     : {c.DRIVE or 'not mounted'}")
print(f"per-take  : {c.EXTRA}")

# The one piece of state that is not in this repo and not on Drive: whether this
# particular machine has been through step 5. Saying it here, in the cell that
# always runs, beats finding out from a tool that exits 2 in a tenth of a second.
venv, ckpt = ROOT / "engine/.venv", ROOT / "engine/checkpoints"
if venv.is_dir() and ckpt.is_dir():
    print("engine    : installed")
else:
    have = [n for n, ok in (("dependencies", venv.is_dir()),
                            ("weights", ckpt.is_dir())) if ok]
    print(f"engine    : NOT READY — run step 5"
          f"{' (have: ' + ', '.join(have) + ')' if have else ''}")

## 4 · A private workspace  *(optional — skip it freely)*

**Skip this cell** and the tool uses this repo as its workspace: the presets in
`presets/`, and `night/collections/example.py` for the batch. You can also just
put your own files there — upload a preset into `presets/`, a lyrics file into
`lyrics/`, a collection module into `night/collections/` — and everything below
will find them.

To use a private content repository instead, this clones it into `private/` and
writes `.workspace`, the one line that points `aimc/workspace.py` at it. None of
your presets or job files need editing: they keep meaning what they meant.

The token comes from **Colab Secrets** (🔑 in the left sidebar): add one named
`GITHUB_TOKEN` holding a fine-grained token with read access to that repository,
and switch on *Notebook access*. It is never printed and never written to disk.

In [ ]:
#@title Clone a private content repo { display-mode: "form" }
PRIVATE_REPO_URL = "https://github.com/Netajam/AI_Music_Creator-songs.git"  #@param {type:"string"}
PRIVATE_BRANCH = "main"  #@param {type:"string"}
USE_PRIVATE = True  #@param {type:"boolean"}

import subprocess
from pathlib import Path

if USE_PRIVATE:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    # The token goes into the URL git is handed and nowhere else: not into the
    # remote it stores, not into a printed line, not onto disk.
    authed = PRIVATE_REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    clone_or_pull(authed, ROOT / "private", PRIVATE_BRANCH)
    subprocess.run(["git", "-C", str(ROOT / "private"), "remote", "set-url",
                    "origin", PRIVATE_REPO_URL], check=True)
    (ROOT / ".workspace").write_text("private\n")
else:
    # Untick to go back to this repo as the workspace, without re-cloning.
    (ROOT / ".workspace").unlink(missing_ok=True)

c = context()
print(f"\nworkspace : {c.WORKSPACE}")
for label, path in (("presets", c.PRESETS), ("lyrics", c.LYRICS),
                    ("collections", c.NIGHT / "collections")):
    n = len(list(path.rglob("*"))) if path.exists() else 0
    print(f"{label:<11} : {path}  ({n} files)")

## 5 · The engine and the weights

Ten minutes, once per machine. [`colab/setup.sh`](colab/setup.sh) does what the
README's install section does on a Mac — clone ACE-Step, `uv sync`, download the
checkpoints — and it is idempotent, so re-running it after a reconnect only
checks. It reports where the time went.

`FULL_CHECKPOINTS` adds the 1.7B LM (3.5 GB). Leave it off unless a preset of
yours names it: `aimc/generation/catalog.py` defaults to the 0.6B.

In [ ]:
#@title Install the engine and download the weights { display-mode: "form" }
FULL_CHECKPOINTS = False  #@param {type:"boolean"}

!bash colab/setup.sh {"--full" if FULL_CHECKPOINTS else ""}

## 6 · What is in the queue

A job file is one take: a preset, a seed, a step count. The runner moves it to
`done/` or `failed/` and appends a line to `ledger.tsv`. That is the whole
protocol, and it is why an interrupted run resumes by starting again — whatever
is still in `queue/` is whatever is still owed.

An empty queue is normal on a fresh clone. This cell says which collections it
can see and how to queue one.

In [ ]:
#@title Queue status { display-mode: "form" }
import json

c = context()
queue = c.NIGHT / "queue"
waiting = sorted(queue.glob("*.json")) if queue.exists() else []


def count(folder: str) -> int:
    d = c.NIGHT / folder
    return len(list(d.glob("*.json"))) if d.exists() else 0


print(f"workspace: {c.WORKSPACE}")
print(f"{len(waiting):>4} waiting   {count('done'):>4} done   "
      f"{count('failed'):>4} failed   {count('held'):>4} held\n")

for job_file in waiting[:10]:
    job = json.loads(job_file.read_text())
    print(f"  {job['collection']:<18} {job['slug']:<28} seed {job['seed']:<4} "
          f"{job['steps']} steps")
if len(waiting) > 10:
    print(f"  … and {len(waiting) - 10} more")

if not waiting:
    colls = sorted(p.stem for p in (c.NIGHT / "collections").glob("*.py")) \
        if (c.NIGHT / "collections").exists() else []
    print("Nothing queued. Collections available in this workspace:")
    print("   ", ", ".join(colls) if colls else "(none — write one, see night/README.md)")
    if colls:
        print(f"\nQueue one with:\n    !python3 night/build.py {colls[0]}")
    print("\nOr skip the batch entirely and use step 9 for a single take.")

## 7 · Render the batch

This starts [`night/batch_render.py`](night/batch_render.py) — the worker that
holds the DiT and the 5Hz LM in memory and walks the queue, instead of paying
three and a half minutes of model loading per take.

**On a Mac that worker was built, measured and reverted.** 16 GiB could not hold
a DiT twice; a take hung for fifty minutes at `[DCW] Built DWT1D` without ever
failing, and `night/runner.sh` — one fresh process per take — is what runs
there. Here the ceiling that decided it is gone, and loading once is the entire
reason to rent a GPU.

It runs detached, so **interrupting this cell does not stop the render**, it
only stops watching. Re-run to pick the watch back up; step 8 stops it.

In [ ]:
#@title Start the worker and watch it { display-mode: "form" }
import os
import subprocess
import time

c = context()
PIDFILE = c.ROOT / "night" / ".colab-worker.pid"
CONSOLE = c.NIGHT / "worker-console.log"   # the very file ledger.tsv names
LEDGER = c.NIGHT / "ledger.tsv"


def worker_pid() -> int | None:
    """The live worker, or None.

    A pid file rather than `pgrep -f batch_render.py`, because `worker.sh` execs
    into `uv run`, which then runs python: the pattern matches two processes and
    signalling the wrong one leaves the other rendering. The pid we wrote is the
    session leader of both, which is the handle we actually want.
    """
    if not PIDFILE.exists():
        return None
    pid = int(PIDFILE.read_text().strip())
    try:
        os.kill(pid, 0)          # signal 0: asks, does not touch
    except OSError:
        return None
    return pid


queue = c.NIGHT / "queue"
waiting = list(queue.glob("*.json")) if queue.exists() else []

if not waiting:
    print("Queue is empty — see step 6. Nothing started.")
elif not engine_ready(c):
    pass
else:
    CONSOLE.parent.mkdir(parents=True, exist_ok=True)
    if worker_pid() is None:
        with CONSOLE.open("ab") as fh:
            # start_new_session: its own process group, so it outlives an
            # interrupted cell — and so step 8 can stop the whole group at once.
            proc = subprocess.Popen(["./night/worker.sh", "--extra", c.EXTRA],
                                    stdout=fh, stderr=fh, start_new_session=True)
        PIDFILE.write_text(str(proc.pid))
        time.sleep(5)
        print(f"worker started (pid {proc.pid}) with: {c.EXTRA}\n")
    else:
        print(f"worker already running (pid {worker_pid()}) — watching it\n")

    # The bar is honest arithmetic, not an estimate: the queue says exactly how
    # many takes are owed, and every take that lands writes its own duration to
    # the ledger. The remaining time is the mean of what this run has actually
    # measured — so it says nothing until the first take is in, and gets more
    # right as it goes.
    total = len(waiting)
    seen = len(LEDGER.read_text().splitlines()) if LEDGER.exists() else 0
    line, started, done, took = Line(), time.time(), 0, []
    synced = 0.0

    try:
        while True:
            rows = LEDGER.read_text().splitlines() if LEDGER.exists() else []
            for row in rows[seen:]:
                if row.startswith("finished_at"):
                    continue
                _when, slug, coll, status, secs, audio, *_ = row.split("\t")
                print(f"  {'ok' if status == 'ok' else '✗ '} {coll}/{slug:<30} "
                      f"{secs:>4}s  {audio}")
                done += 1
                if status == "ok":
                    took.append(int(secs))
            seen = len(rows)

            left = len(list(queue.glob("*.json")))
            eta = f", ~{clock(sum(took) / len(took) * left)} left" if took else ""
            line(f"  {bar(done, total)}  {done}/{total} takes, "
                 f"{clock(time.time() - started)} elapsed{eta}")

            # Drive is the slow thing here (a FUSE mount, tens of MB/s), so it
            # is not on the same beat as the bar.
            if time.time() - synced > 60:
                to_drive(c)
                synced = time.time()

            if worker_pid() is None:
                to_drive(c)
                # Before the first take lands there is nothing in the ledger and
                # nothing to see but the models loading; after a crash the same
                # is true and the reason is in the console. Show it either way.
                if CONSOLE.exists():
                    print("\n".join(
                        CONSOLE.read_text(errors="replace").splitlines()[-12:]))
                print(f"\nworker stopped — {left} job(s) still queued")
                break
            time.sleep(5)
    except KeyboardInterrupt:
        print(f"\nstopped watching. The worker (pid {worker_pid()}) is still "
              f"rendering — re-run this cell to watch it again.")

## 8 · Stop the worker

In [ ]:
#@title Stop after the take in flight { display-mode: "form" }
# SIGTERM to the whole process group: worker.sh, uv and python are all in it,
# and killing only the first would leave the third rendering. The take in flight
# is lost either way — but its job file is still in queue/, because a job only
# moves once its wav exists, so the next run picks it up untouched.
import os
import signal

c = context()
pidfile = c.ROOT / "night" / ".colab-worker.pid"
if pidfile.exists():
    pid = int(pidfile.read_text().strip())
    try:
        os.killpg(os.getpgid(pid), signal.SIGTERM)
        print(f"sent SIGTERM to the worker's process group ({pid})")
    except OSError as exc:
        print(f"nothing to stop: {exc}")
    pidfile.unlink()
else:
    print("no worker running")

## 9 · A single take

The same `./song` the README documents, unchanged — and the quickest way to try
something without touching the queue at all.

`PRESET` is relative to the working directory: `presets/electro-house.json` for
one that ships here, `private/presets/…` for one of your own. Leave it empty to
use `STYLE` instead.

To use a track as a style reference, fetch it first — `./grab "<youtube url>"`,
then `./blend-refs refs/downloads/<file>=45 -o refs/melange.wav` — and add
`"reference": "../refs/melange.wav"` to a preset. `--reference` takes a path,
never a link.

In [ ]:
#@title Render one song { display-mode: "form" }
PRESET = "presets/electro-house.json"  #@param {type:"string"}
STYLE = ""  #@param {type:"string"}
LYRICS_FILE = ""  #@param {type:"string"}
INSTRUMENTAL = False  #@param {type:"boolean"}
SEED = 1  #@param {type:"integer"}
STEPS = 8  #@param {type:"integer"}

c = context()
argv = ["./song", "--seed", str(SEED), "--steps", str(STEPS),
        "--out", str(c.SONGS / "colab")]
if PRESET:
    argv += ["--preset", PRESET]
if STYLE:
    argv += ["--style", STYLE]
if LYRICS_FILE:
    argv += ["--lyrics", LYRICS_FILE]
if INSTRUMENTAL:
    argv += ["--instrumental"]
argv += shlex.split(c.EXTRA)

if engine_ready(c):
    run_song(argv, expect=expected_seconds(c, STEPS))

## 10 · Write the words

Lyrics for this engine are not free-form. `docs/recipes.md` learned the rules by
rendering, and three of them are ones every chat model breaks unprompted: it
invents `[Pre-Chorus]` and `[guitar solo]`, which the 5Hz LM reads as **sections
of their own** so the music stops; it rewords the chorus on each pass for
variety, which is exactly what stops a song sticking; and it has no idea that
the character count has to match the duration.

So this step does not generate anything. The first cell writes a **prompt**
carrying those rules, for whichever model you already use — Claude, ChatGPT,
anything. The second runs [`night/build.py`](night/build.py)'s own checker over
what comes back — the same one that refuses to queue a batch job, so a lyric
that passes here passes there — and saves it under `lyrics/colab/`.

Keeping generation outside is deliberate. A local model would fight the DiT and
the 5Hz LM for the same VRAM — step 7's note on the Mac is that war, already
lost once — and it would write worse French than the model you can already
open in another tab.


In [ ]:
#@title 10a · Build the prompt, paste it into a chat model { display-mode: "form" }
THEME = "une nuit qui ne finit pas dans une ville fermée"  #@param {type:"string"}
LANGUAGE = "French"  #@param ["French", "English", "Spanish", "Italian", "German", "Portuguese", "Dutch", "Japanese"]
STRUCTURE = "pop"  #@param ["pop", "electro"]
DURATION = 180  #@param {type:"slider", min:60, max:300, step:5}
DELIVERY = "sung"  #@param ["sung", "spoken"]

#@markdown `pop` is Intro / Verse / Chorus / Verse / Chorus / Bridge / Chorus /
#@markdown Outro. `electro` swaps the chorus for Build / Drop and puts a
#@markdown Breakdown in the middle. Both are eight sections, which is what the
#@markdown book says three minutes wants.

print(lyrics_prompt(THEME, LANGUAGE, STRUCTURE, DURATION, DELIVERY))


In [ ]:
#@title 10b · Check what came back, and save it { display-mode: "form" }

# Paste the model's answer between the triple quotes, replacing this draft.
LYRICS = """
[Intro]

[Verse]

[Chorus]

[Verse]

[Chorus]

[Bridge]

[Chorus]

[Outro]
"""

NAME = "ma-chanson"  #@param {type:"string"}
DURATION = 180  #@param {type:"slider", min:60, max:300, step:5}
SPARSE = False  #@param {type:"boolean"}
SAVE_ANYWAY = False  #@param {type:"boolean"}

#@markdown This runs **`night/build.py`'s own checker** — the same one that
#@markdown refuses to queue a batch job — so a lyric that passes here passes
#@markdown there. Tick `SPARSE` for a song meant to be bare (footwork, kwaito,
#@markdown early dubstep): it lowers the density floor rather than padding the
#@markdown words out of shape. `SAVE_ANYWAY` overrides everything else.

c = context()
problems = night_build().check_lyrics(NAME, LYRICS, DURATION, sparse=SPARSE)

for problem in problems:
    print(f"  ✗ {problem}")
if not problems:
    print("  ✓ tags, sections, chorus and density all check out")

if problems and not SAVE_ANYWAY:
    print(f"\n{len(problems)} problem(s) — nothing saved. Fix the lyrics above, or "
          f"tick SAVE_ANYWAY.")
else:
    target = c.LYRICS / "colab" / f"{NAME}.txt"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(LYRICS.strip() + "\n")
    rel = target.relative_to(c.WORKSPACE)
    print(f"\nsaved {rel}")
    print(f"Use it in step 9 by putting this in LYRICS_FILE:\n    {rel}")


## 11 · Compose a song from scratch

Every setting, filled in here — no preset file to write first.

The cell **saves what you filled in** before it renders: a lyrics file and a
preset, under `lyrics/colab/` and `presets/colab/` in the workspace. That is not
bookkeeping for its own sake. A take you like is worth nothing if you cannot say
what produced it, and a JSON preset plus a seed is the whole of that — re-run
the same preset with the same seed and you get the same song back.

The code is left visible on purpose: `LYRICS` is a normal Python string and you
edit it in place. Everything below it is a form field.

Only eight tags are read as sections — `Intro`, `Verse`, `Chorus`, `Build`,
`Drop`, `Breakdown`, `Bridge`, `Outro`. Anything else in brackets is read as a
section of its own rather than as a stage direction, which is a mistake this
project has paid for more than once; see [`docs/recipes.md`](docs/recipes.md).

In [ ]:
#@title Compose a song from scratch { run: "auto" }

# ── the words ────────────────────────────────────────────────────────────────
# Edit this directly. Ignored when INSTRUMENTAL is ticked.
LYRICS = """
[Intro]

[Verse]
On a fermé la ville à double tour
Les néons comptent les secondes
Personne ne rentre avant le jour
Personne ne dort quand ça gronde

[Build]
Et ça monte, et ça monte
Le sol tremble sous les pas

[Drop]
LÂCHE TOUT, LA NUIT NOUS APPARTIENT
LÂCHE TOUT, ON NE RENTRERA PAS

[Outro]
"""

# ── what it is ───────────────────────────────────────────────────────────────
NAME = "ma-chanson"  #@param {type:"string"}
STYLE = "Energetic French electro-house, four-on-the-floor kick, sidechained bassline, dark minor-key synth hook, sung-spoken male vocal, festival energy"  #@param {type:"string"}
NEGATIVE = "acoustic, ballad, slow, orchestral, female lead vocals, lo-fi, muddy mix"  #@param {type:"string"}
LANGUAGE = "fr"  #@param ["fr", "en", "es", "it", "de", "pt", "nl", "ja"]
INSTRUMENTAL = False  #@param {type:"boolean"}

# ── the music ────────────────────────────────────────────────────────────────
BPM = 126  #@param {type:"integer"}
KEY = "D minor"  #@param {type:"string"}
DURATION = 145  #@param {type:"number"}

# ── a track to imitate the style of (optional) ───────────────────────────────
# A path, never a link. ./grab "<youtube url>" is what turns one into the other.
REFERENCE = ""  #@param {type:"string"}
STYLE_STRENGTH = 0.4  #@param {type:"slider", min:0, max:1, step:0.05}

# ── the levers ───────────────────────────────────────────────────────────────
SEED = 1  #@param {type:"integer"}
STEPS = 8  #@param {type:"slider", min:4, max:32, step:1}
LM_TEMPERATURE = 0.85  #@param {type:"slider", min:0.1, max:1.5, step:0.05}
LM_CFG = 2.2  #@param {type:"slider", min:1, max:5, step:0.1}
FADE_OUT = 4  #@param {type:"number"}
RENDER = True  #@param {type:"boolean"}

import json
import re
import shlex
import subprocess

c = context()
slug = re.sub(r"[^a-z0-9-]+", "-", NAME.lower()).strip("-") or "sans-titre"

# --style is rejected at 512 characters, but only after the preset has loaded.
# Two seconds here beats finding out once the models are in memory.
if len(STYLE) >= 512:
    raise SystemExit(f"style is {len(STYLE)} characters; the limit is 512")

preset = {
    "style": STYLE,
    "negative": NEGATIVE,
    "language": LANGUAGE,
    "bpm": BPM,
    "key": KEY,
    "duration": DURATION,
    "lm_temperature": LM_TEMPERATURE,
    "lm_cfg": LM_CFG,
    "fade_out": FADE_OUT,
}

pre_dir = c.PRESETS / "colab"
lyr_dir = c.LYRICS / "colab"
pre_dir.mkdir(parents=True, exist_ok=True)

if INSTRUMENTAL:
    preset["instrumental"] = True
else:
    lyr_dir.mkdir(parents=True, exist_ok=True)
    (lyr_dir / f"{slug}.txt").write_text(LYRICS.strip() + "\n", encoding="utf-8")
    # Relative to the preset, which is how every other preset in this repo says
    # it, so the pair can be moved or committed together.
    preset["lyrics"] = f"../../lyrics/colab/{slug}.txt"

if REFERENCE:
    preset["reference"] = REFERENCE
    preset["style_strength"] = STYLE_STRENGTH

preset_path = pre_dir / f"{slug}.json"
preset_path.write_text(json.dumps(preset, ensure_ascii=False, indent=2) + "\n",
                       encoding="utf-8")
print(f"saved  {preset_path.relative_to(c.WORKSPACE)}")
if not INSTRUMENTAL:
    print(f"saved  {(lyr_dir / f'{slug}.txt').relative_to(c.WORKSPACE)}")

argv = ["./song", "--preset", str(preset_path), "--seed", str(SEED),
        "--steps", str(STEPS), "--out", str(c.SONGS / "colab")]
argv += shlex.split(c.EXTRA)

if not RENDER:
    print("\nRENDER is off — the preset above is written, nothing was generated.")
elif engine_ready(c):
    print()
    run_song(argv, expect=expected_seconds(c, STEPS))

In [ ]:
#@title Listen to the newest take { display-mode: "form" }
from IPython.display import Audio, display

c = context()
takes = finished_takes(c)
if not takes:
    print(f"nothing rendered yet under {c.SONGS}")
else:
    newest = takes[-1]
    print(f"{newest.relative_to(c.WORKSPACE)}  "
          f"({newest.stat().st_size / 1024**2:.1f} MB)")
    display(Audio(str(newest)))

## 12 · A style reference

`reference` in a preset is **a path to an audio file, never a link**. The engine
reads a file, and it reads only 30 seconds of it: `process_reference_audio`
takes three 10-second segments and stitches them, so what you point at should
already be the part worth imitating.

This cell makes that file. Two ways in:

- **Upload one you have** — leave `SOURCE` empty. Always works, no third party
  involved, and the obvious choice for anything you own.
- **Fetch from YouTube** — put a URL or a search in `SOURCE`. Note that this is
  against YouTube's terms of service and the audio stays under copyright unless
  it is yours, public domain or freely licensed; the repo says so too, in
  `aimc/references/grab.py`. Note also that YouTube frequently refuses
  datacentre addresses, so this fails on Colab more often than it does at home —
  when it does, upload instead.

`./blend-refs` can stitch **three** tracks into one reference, each at a chosen
moment, giving the model three influences of equal weight:
`./blend-refs a.mp3=45 b.mp3=30 c.mp3=12 -o refs/melange.wav`.

In [ ]:
#@title Fetch or upload a style reference { display-mode: "form" }
SOURCE = ""  #@param {type:"string"}
AT_SECOND = 45  #@param {type:"integer"}
NAME = "melange"  #@param {type:"string"}

import shutil
import subprocess

c = context()
downloads = c.REFS / "downloads"
downloads.mkdir(parents=True, exist_ok=True)
before = set(downloads.iterdir())

if SOURCE:
    if shutil.which("yt-dlp") is None:
        print("installing yt-dlp…")
        subprocess.run(["pip", "install", "-q", "yt-dlp"], check=False)
    # ./grab takes a URL as-is, or anything else as a YouTube search.
    subprocess.run(["./grab", SOURCE], check=False)
else:
    from google.colab import files

    print("Choose an audio file to upload…")
    for name, blob in files.upload().items():
        (downloads / name).write_bytes(blob)

new = sorted(set(downloads.iterdir()) - before,
             key=lambda f: f.stat().st_mtime)
if not new:
    print("\nNothing arrived in refs/downloads — nothing to blend.")
else:
    got = new[-1]
    print(f"\ngot {got.name}")
    out = c.REFS / f"{NAME}.wav"
    # `<file>=<second>` is where in the track to take the ten seconds from: aim
    # at a chorus or a drop, never an intro.
    run_song(["./blend-refs", f"{got}={AT_SECOND}", "-o", str(out)])
    if out.exists():
        print(f"\nreference ready: {out}")
        print(f'  in a preset  : "reference": "{out}"')
        print(f"  or step 10   : REFERENCE = {out}")

## 13 · Rework a take you already have

Four ways to change a take rather than re-roll the dice, and the recipe book is
emphatic that they beat new seeds. They are mutually exclusive — the CLI says so
itself — because they answer different questions:

| mode | what it does | what it needs |
|---|---|---|
| `retake` | the same song, moved a little | the **preset** and the original **seed** |
| `repaint` | one passage regenerated, the rest preserved | the **take**, and a time range |
| `cover` | the whole track re-recorded in another style | the **take**, and a new style |
| `lego` | one instrument track added to it | the **take**, and which track |

`retake` is the odd one: it varies a *seed*, so it takes the preset rather than
the audio. `0.1–0.2` is the same track slightly different; `0.5+` is a
reinterpretation.

Only `lego` reads the source take's **manifest** — the settings file written
beside every wav — inheriting its tempo, key, time signature and lyrics, because
adding a guitar means playing *inside* that track and a part that does not know
the grid lands beside it rather than on it. `inherit_from_source` returns
immediately for every other mode, so `repaint` and `cover` take their tempo and
their words from the preset you give them, not from the take.

Leave `SOURCE_TAKE` empty to list what this workspace has.

In [ ]:
#@title Rework a take { display-mode: "form" }
MODE = "repaint"  #@param ["retake", "repaint", "cover", "lego"]
SOURCE_TAKE = ""  #@param {type:"string"}
PRESET = "presets/electro-house.json"  #@param {type:"string"}

#@markdown --- retake
RETAKE_SEED = 1  #@param {type:"integer"}
RETAKE_VARIANCE = 0.15  #@param {type:"slider", min:0.05, max:1.0, step:0.05}

#@markdown --- repaint (seconds; TO of -1 means "to the end")
REPAINT_FROM = 78  #@param {type:"number"}
REPAINT_TO = 92  #@param {type:"number"}
REPAINT_MODE = "balanced"  #@param ["conservative", "balanced", "aggressive"]

#@markdown --- cover
COVER_STYLE = ""  #@param {type:"string"}

#@markdown --- lego
LEGO_TRACK = "drums"  #@param ["drums", "bass", "guitar", "keyboard", "percussion", "strings", "synth", "brass", "woodwinds", "fx", "vocals", "backing_vocals"]

#@markdown ---
SEED = 1  #@param {type:"integer"}
STEPS = 8  #@param {type:"slider", min:4, max:32, step:1}

import shlex

c = context()

if not SOURCE_TAKE and MODE != "retake":
    takes = finished_takes(c)[::-1]      # newest first
    print(f"Takes in {c.SONGS}:" if takes else f"No takes yet under {c.SONGS}.")
    for t in takes[:12]:
        beside = "settings" if t.with_suffix(".json").exists() else "no manifest"
        print(f"  {t.relative_to(c.WORKSPACE)}   ({beside})")
    print("\nPut one of those in SOURCE_TAKE and run again.")
else:
    argv = ["./song", "--seed", str(SEED), "--steps", str(STEPS),
            "--out", str(c.SONGS / "colab")]
    if MODE == "retake":
        # A variation on a seed, so it needs the preset that made it, not the wav.
        argv += ["--preset", PRESET, "--retake-seed", str(RETAKE_SEED),
                 "--retake-variance", str(RETAKE_VARIANCE)]
    elif MODE == "repaint":
        argv += ["--preset", PRESET, "--repaint", SOURCE_TAKE,
                 "--repaint-from", str(REPAINT_FROM), "--repaint-to", str(REPAINT_TO),
                 "--repaint-mode", REPAINT_MODE]
    elif MODE == "cover":
        # The preset comes too, and not only for its style: --cover still needs
        # lyrics, and unlike --lego it inherits nothing from the source take.
        # COVER_STYLE then overrides the preset's style, as any option does.
        argv += ["--preset", PRESET, "--cover", SOURCE_TAKE]
        if COVER_STYLE:
            argv += ["--style", COVER_STYLE]
    elif MODE == "lego":
        argv += ["--preset", PRESET, "--lego", SOURCE_TAKE,
                 "--lego-track", LEGO_TRACK]
    argv += shlex.split(c.EXTRA)

    if engine_ready(c):
        run_song(argv, expect=expected_seconds(c, STEPS))

## 14 · Bring it home

With Drive mounted, step 7 has been copying as it went and this only catches the
last take. Without Drive, this is the one chance to get the audio off the
machine before it is reclaimed.

The takes do not come back through git — `songs/` is gitignored, deliberately.
What is worth committing is `night/ledger.tsv` and the job files that moved into
`night/done/`: the record of what was rendered, and the only part of a run that
re-running cannot reproduce.

In [ ]:
#@title Sync to Drive, or build a zip { display-mode: "form" }
import subprocess

c = context()
if c.DRIVE is not None:
    to_drive(c)
    takes = list((c.DRIVE / "songs").rglob("*.wav"))
    print(f"{c.DRIVE}  —  {len(takes)} takes, "
          f"{sum(f.stat().st_size for f in takes) / 1024**3:.2f} GB")
else:
    archive = "/content/takes.zip"
    paths = [str(p) for p in (c.SONGS, c.NIGHT / "ledger.tsv", c.NIGHT / "done")
             if p.exists()]
    if not paths:
        print("nothing to take home yet")
    else:
        subprocess.run(["zip", "-r", "-q", archive, *paths], check=False)
        from google.colab import files

        files.download(archive)